# EDA — Treatment vs Disease Progression (6 m)

## 🎯 Objective
Descriptive exploration of whether there is an association between the available treatment information and the rate of functional progression (ALSFRS-R slope), including:
- **Study arm** (Active vs Placebo) — clinical trial arm
- **Riluzole** — the only FDA/EMA-approved drug for ALS at the time of the PRO-ACT trials
- **Concomitant medication** — number and type of pre-baseline medications
- **Interactions** — study_arm × riluzole combinations

## 📥 Input
- `01_data/processed/dataset_6m_v2.csv` (1 row per patient, DEV + TEST)
- `01_data/raw/PROACT_CONMEDS.csv` (concomitant medication records)

## 📤 Outputs
- `04_outputs/figures/eda_treatment/eda_treat_arm_slope_dist.png`
- `04_outputs/figures/eda_treatment/eda_treat_arm_rapid_rate.png`
- `04_outputs/figures/eda_treatment/eda_treat_riluzole_slope_dist.png`
- `04_outputs/figures/eda_treatment/eda_treat_interaction.png`
- `04_outputs/figures/eda_treatment/eda_treat_top_meds.png`
- `04_outputs/tables/eda_treatment/eda_treat_summary.csv`

<div style="padding:10px;border-left:6px solid #FF5F5D;">
<b>Note:</b> this analysis is <b>descriptive</b> (observational). PRO-ACT aggregates multiple clinical trials,
so "Active" may represent different drugs depending on the trial.
It is not possible to infer causality ("treatment X causes slow progression")
without causal inference techniques (propensity score matching, IPTW, etc.).
</div>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, "font.size": 11})

SEED       = 42
RAPID_FRAC = 0.30

# ── Paths ──
ROOT       = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROCESSED  = os.path.join(ROOT, "01_data", "processed")
RAW        = os.path.join(ROOT, "01_data", "raw")
OUT_FIG    = os.path.join(ROOT, "04_outputs", "figures", "eda_treatment")
OUT_TAB    = os.path.join(ROOT, "04_outputs", "tables",  "eda_treatment")
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_TAB, exist_ok=True)

print("Paths OK")

## 1) Data Loading and Preparation

We load the v2 dataset (6 m) and define the `rapid` label using the global 30th percentile of slope.
This definition is used **for visual reference only** — in the training pipeline, the threshold is computed fold-wise.

In [ ]:
df = pd.read_csv(os.path.join(PROCESSED, "dataset_6m_v2.csv"))
slope_col = "slope_180d_per_30d"
df[slope_col] = pd.to_numeric(df[slope_col], errors="coerce")

# Global binary label (reference only)
thr = np.nanquantile(df[slope_col].values, RAPID_FRAC)
df["rapid"] = (df[slope_col] <= thr).astype(int)

print(f"N = {len(df)} patients")
print(f"Rapid threshold (P30): {thr:.3f} points/30d")
print(f"Rapid prevalence: {df.rapid.mean():.1%}")
print()
print("Available treatment columns:")
for c in ["riluzole_pre_t0", "study_arm", "n_conmeds_pre_t0"]:
    print(f"  {c}: {df[c].nunique()} unique values, NaN={df[c].isna().sum()}")

## 2) Study Arm: Active vs Placebo

In PRO-ACT, each patient belongs to a clinical trial arm:
- **Active** — received the experimental drug from the respective trial
- **Placebo** — received placebo

⚠️ **"Active" is NOT a single drug** — it is the experimental arm of multiple different trials (lithium, dexpramipexole, ceftriaxone, etc.).
This dilutes any real pharmacological signal.

In [ ]:
# ── 2a) Descriptive table: Study Arm ──
arm_stats = []
for arm in ["Active", "Placebo"]:
    sub = df[df.study_arm == arm]
    arm_stats.append({
        "Group": arm,
        "N": len(sub),
        "Rapid (%)": f"{100*sub.rapid.mean():.1f}",
        "Slope mean": f"{sub[slope_col].mean():.3f}",
        "Slope std":  f"{sub[slope_col].std():.3f}",
        "Slope median": f"{sub[slope_col].median():.3f}",
    })
sub_na = df[df.study_arm.isna()]
arm_stats.append({
    "Group": "NaN (no info)",
    "N": len(sub_na),
    "Rapid (%)": f"{100*sub_na.rapid.mean():.1f}",
    "Slope mean": f"{sub_na[slope_col].mean():.3f}",
    "Slope std":  f"{sub_na[slope_col].std():.3f}",
    "Slope median": f"{sub_na[slope_col].median():.3f}",
})
arm_df = pd.DataFrame(arm_stats)
display(arm_df)

In [ ]:
# ── 2b) Slope distribution by Study Arm ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Histogram
ax = axes[0]
for arm, color in [("Active", "#2196F3"), ("Placebo", "#FF5722")]:
    sub = df[df.study_arm == arm]
    ax.hist(sub[slope_col], bins=40, alpha=0.55, label=f"{arm} (n={len(sub)})",
            color=color, edgecolor="white", linewidth=0.5)
ax.axvline(thr, color="red", linestyle="--", linewidth=1.2, label=f"P30 = {thr:.2f}")
ax.set_xlabel("Slope (ALSFRS-R points / 30 days)")
ax.set_ylabel("Frequency")
ax.set_title("Slope Distribution by Study Arm")
ax.legend(fontsize=9)

# Box plot
ax = axes[1]
data_box = df[df.study_arm.notna()].copy()
order = ["Active", "Placebo"]
bp = sns.boxplot(data=data_box, x="study_arm", y=slope_col, order=order,
                 palette=["#2196F3", "#FF5722"], ax=ax, width=0.5, fliersize=3)
# Annotate medians
for i, arm in enumerate(order):
    med = data_box[data_box.study_arm == arm][slope_col].median()
    ax.text(i, med + 0.05, f"med={med:.2f}", ha="center", fontsize=9, fontweight="bold")
ax.axhline(thr, color="red", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("Study Arm")
ax.set_ylabel("Slope (points / 30 days)")
ax.set_title("Slope by Study Arm (boxplot)")

fig.suptitle("Study Arm vs ALSFRS-R Slope (6 m)", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT_FIG, "eda_treat_arm_slope_dist.png"), bbox_inches="tight")
plt.show()
print("Saved:", os.path.join(OUT_FIG, "eda_treat_arm_slope_dist.png"))

In [ ]:
# ── 2c) Rapid progression rate by Study Arm ──
fig, ax = plt.subplots(figsize=(6, 4))

groups = ["Active", "Placebo", "NaN"]
rates  = []
counts = []
for g in groups:
    sub = df[df.study_arm == g] if g != "NaN" else df[df.study_arm.isna()]
    rates.append(100 * sub.rapid.mean())
    counts.append(len(sub))

colors = ["#2196F3", "#FF5722", "#9E9E9E"]
bars = ax.bar(groups, rates, color=colors, edgecolor="black", linewidth=0.6, width=0.55)

# Annotate bars
for bar, rate, n in zip(bars, rates, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f"{rate:.1f}%\n(n={n})", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.axhline(100 * RAPID_FRAC, color="red", linestyle="--", linewidth=1,
           label=f"Global prevalence ({100*RAPID_FRAC:.0f}%)")
ax.set_ylabel("Rapid Progression Rate (%)")
ax.set_title("Rapid progression rate by Study Arm", fontweight="bold")
ax.set_ylim(0, max(rates) + 10)
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(OUT_FIG, "eda_treat_arm_rapid_rate.png"), bbox_inches="tight")
plt.show()
print("Saved:", os.path.join(OUT_FIG, "eda_treat_arm_rapid_rate.png"))

## 3) Riluzole

Riluzole is the only FDA/EMA-approved drug for ALS (at the time of the PRO-ACT trials).
The feature `riluzole_pre_t0` indicates whether the patient was taking riluzole **before** baseline.

⚠️ Riluzole use is not randomised — there may be **confounders** (patients taking riluzole may have systematically different clinical profiles).

In [ ]:
# ── 3a) Slope distribution by Riluzole ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for ril, label, color in [(0, "No riluzole", "#FF9800"), (1, "Riluzole", "#4CAF50")]:
    sub = df[df.riluzole_pre_t0 == ril]
    ax.hist(sub[slope_col], bins=40, alpha=0.55, label=f"{label} (n={len(sub)})",
            color=color, edgecolor="white", linewidth=0.5)
ax.axvline(thr, color="red", linestyle="--", linewidth=1.2, label=f"P30 = {thr:.2f}")
ax.set_xlabel("Slope (ALSFRS-R points / 30 days)")
ax.set_ylabel("Frequency")
ax.set_title("Slope Distribution by Riluzole")
ax.legend(fontsize=9)

ax = axes[1]
ril_labels = ["No riluzole", "Riluzole"]
ril_rates = []
ril_counts = []
ril_slopes = []
for ril in [0, 1]:
    sub = df[df.riluzole_pre_t0 == ril]
    ril_rates.append(100 * sub.rapid.mean())
    ril_counts.append(len(sub))
    ril_slopes.append(sub[slope_col].median())

colors_ril = ["#FF9800", "#4CAF50"]
bars = ax.bar(ril_labels, ril_rates, color=colors_ril, edgecolor="black",
              linewidth=0.6, width=0.5)
for bar, rate, n, med in zip(bars, ril_rates, ril_counts, ril_slopes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f"{rate:.1f}%\n(n={n})\nmed={med:.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.axhline(100 * RAPID_FRAC, color="red", linestyle="--", linewidth=1,
           label=f"Global prevalence ({100*RAPID_FRAC:.0f}%)")
ax.set_ylabel("Rapid Progression Rate (%)")
ax.set_title("Rapid progression rate by Riluzole")
ax.set_ylim(0, max(ril_rates) + 12)
ax.legend(fontsize=9)

fig.suptitle("Riluzole vs ALSFRS-R Slope (6 m)", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT_FIG, "eda_treat_riluzole_slope_dist.png"), bbox_inches="tight")
plt.show()
print("Saved:", os.path.join(OUT_FIG, "eda_treat_riluzole_slope_dist.png"))

## 4) Interaction: Study Arm × Riluzole

We cross the two treatment variables to check for any interaction pattern.
For example: do patients in the Active arm who **also** take riluzole show different progression?

In [ ]:
# ── 4a) Cross-tabulation ──
cross_data = []
for arm in ["Active", "Placebo"]:
    for ril in [0, 1]:
        sub = df[(df.study_arm == arm) & (df.riluzole_pre_t0 == ril)]
        if len(sub) > 5:
            cross_data.append({
                "Study Arm": arm,
                "Riluzole": "Yes" if ril else "No",
                "N": len(sub),
                "Rapid (%)": f"{100*sub.rapid.mean():.1f}",
                "Slope mean": f"{sub[slope_col].mean():.3f}",
                "Slope median": f"{sub[slope_col].median():.3f}",
            })
cross_df = pd.DataFrame(cross_data)
display(cross_df)

In [ ]:
# ── 4b) Grouped bar chart ──
fig, ax = plt.subplots(figsize=(8, 5))

x_labels = [f"{row['Study Arm']}\n{row['Riluzole']}" for _, row in cross_df.iterrows()]
rates_cross = [float(row["Rapid (%)"]) for _, row in cross_df.iterrows()]
ns_cross = [int(row["N"]) for _, row in cross_df.iterrows()]

# Colors: Active-No, Active-Yes, Placebo-No, Placebo-Yes
colors_cross = ["#90CAF9", "#1565C0", "#FFAB91", "#D84315"]
bars = ax.bar(range(len(x_labels)), rates_cross, color=colors_cross,
              edgecolor="black", linewidth=0.6, width=0.6)

for bar, rate, n in zip(bars, rates_cross, ns_cross):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.6,
            f"{rate:.1f}%\n(n={n})", ha="center", va="bottom",
            fontsize=9, fontweight="bold")

ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, fontsize=10)
ax.axhline(100 * RAPID_FRAC, color="red", linestyle="--", linewidth=1,
           label=f"Global prevalence ({100*RAPID_FRAC:.0f}%)")
ax.set_ylabel("Rapid Progression Rate (%)")
ax.set_title("Interaction: Study Arm × Riluzole", fontweight="bold")
ax.set_ylim(0, max(rates_cross) + 12)
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(OUT_FIG, "eda_treat_interaction.png"), bbox_inches="tight")
plt.show()
print("Saved:", os.path.join(OUT_FIG, "eda_treat_interaction.png"))

## 5) Concomitant Medication: Quantity and Top Drugs

We analyse:
- The **number of concomitant medications** pre-baseline (`n_conmeds_pre_t0`) vs progression
- The **most frequent drugs** in the cohort and their association with the rapid progression rate

We load the raw `PROACT_CONMEDS.csv` file to obtain individual medication names.

In [ ]:
# ── 5a) n_conmeds_pre_t0 vs slope ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Scatter
ax = axes[0]
ax.scatter(df["n_conmeds_pre_t0"], df[slope_col], alpha=0.25, s=15, color="#607D8B")
ax.axhline(thr, color="red", linestyle="--", linewidth=1, alpha=0.7, label=f"P30 = {thr:.2f}")
ax.set_xlabel("No. concomitant medications pre-baseline")
ax.set_ylabel("Slope (points / 30 days)")
ax.set_title("Slope vs No. Conmeds")
ax.legend(fontsize=9)

# Binned rapid rate
ax = axes[1]
bins = [0, 1, 3, 6, 10, np.inf]
labels_bin = ["0", "1-2", "3-5", "6-9", "10+"]
df["conmeds_bin"] = pd.cut(df["n_conmeds_pre_t0"], bins=bins, labels=labels_bin, right=False)

bin_data = df.groupby("conmeds_bin", observed=True).agg(
    n=("rapid", "count"),
    rapid_rate=("rapid", "mean"),
).reset_index()

bars = ax.bar(bin_data["conmeds_bin"].astype(str), 100 * bin_data["rapid_rate"],
              color="#78909C", edgecolor="black", linewidth=0.6, width=0.6)
for bar, _, row in zip(bars, range(len(bin_data)), bin_data.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.6,
            f"{100*row.rapid_rate:.1f}%\n(n={row.n})",
            ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax.axhline(100 * RAPID_FRAC, color="red", linestyle="--", linewidth=1,
           label=f"Global prevalence ({100*RAPID_FRAC:.0f}%)")
ax.set_xlabel("No. concomitant medications (bins)")
ax.set_ylabel("Rapid Progression Rate (%)")
ax.set_title("Rapid rate by conmeds group")
ax.set_ylim(0, 100 * bin_data["rapid_rate"].max() + 12)
ax.legend(fontsize=9)

fig.suptitle("Concomitant Medication vs Progression (6 m)", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── 5b) Top 15 medications and rapid progression rate ──
cohort_ids = set(df.subject_id)
cm = pd.read_csv(os.path.join(RAW, "PROACT_CONMEDS.csv"))

# Pre-baseline records from our cohort only
cm["Start_Delta"] = pd.to_numeric(cm["Start_Delta"], errors="coerce")
cm = cm.merge(df[["subject_id", "t0_delta_days"]], on="subject_id", how="inner")
cm_pre = cm[cm["Start_Delta"] <= cm["t0_delta_days"]].copy()
print(f"Pre-baseline records in cohort: {len(cm_pre)}")
print(f"Patients with ≥1 pre-baseline medication: {cm_pre.subject_id.nunique()}")

# Top 15 by number of subjects
med_counts = cm_pre.groupby("Medication_Coded")["subject_id"].nunique()
top15 = med_counts.sort_values(ascending=False).head(15)

# For each, compute rapid rate
med_stats = []
for med_name, n_subj in top15.items():
    subj_ids = set(cm_pre[cm_pre.Medication_Coded == med_name].subject_id)
    sub = df[df.subject_id.isin(subj_ids)]
    med_stats.append({
        "Medication": med_name,
        "N_subjects": n_subj,
        "Pct_cohort": round(100 * n_subj / len(cohort_ids), 1),
        "Rapid_rate": round(100 * sub.rapid.mean(), 1) if len(sub) > 0 else 0,
        "Slope_median": round(sub[slope_col].median(), 3) if len(sub) > 0 else 0,
    })
med_stats_df = pd.DataFrame(med_stats)
display(med_stats_df)

In [ ]:
# ── 5c) Horizontal chart: Top 15 meds × rapid rate ──
fig, ax = plt.subplots(figsize=(9, 6))

plot_df = med_stats_df.sort_values("Rapid_rate", ascending=True)
y_pos = range(len(plot_df))

colors_med = ["#D84315" if r > 100*RAPID_FRAC else "#4CAF50"
              for r in plot_df["Rapid_rate"]]

bars = ax.barh(y_pos, plot_df["Rapid_rate"], color=colors_med,
               edgecolor="black", linewidth=0.5, height=0.65)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{m}  (n={n})" for m, n in
                     zip(plot_df["Medication"], plot_df["N_subjects"])], fontsize=9)

# Annotate values
for bar, rate in zip(bars, plot_df["Rapid_rate"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{rate:.1f}%", va="center", fontsize=9, fontweight="bold")

ax.axvline(100 * RAPID_FRAC, color="red", linestyle="--", linewidth=1.2,
           label=f"Global prevalence ({100*RAPID_FRAC:.0f}%)")
ax.set_xlabel("Rapid Progression Rate (%)")
ax.set_title("Top 15 pre-baseline medications × Rapid Progression Rate",
             fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0, max(plot_df["Rapid_rate"]) + 8)

fig.tight_layout()
fig.savefig(os.path.join(OUT_FIG, "eda_treat_top_meds.png"), bbox_inches="tight")
plt.show()
print("Saved:", os.path.join(OUT_FIG, "eda_treat_top_meds.png"))

## 6) Statistical Tests: Study Arm × Progression

We perform a χ² (chi-squared) test to check whether the proportion of rapid progressors
differs significantly between the Active and Placebo arms.

Additionally, a Mann-Whitney U test is used to compare the slope distributions.

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu

# ── 6a) Chi-squared: Study Arm vs Rapid ──
sub_known = df[df.study_arm.notna()].copy()
ct = pd.crosstab(sub_known["study_arm"], sub_known["rapid"])
chi2, p_chi, dof, expected = chi2_contingency(ct)
print("Crosstab (Study Arm × Rapid):")
display(ct)
print(f"\nχ² = {chi2:.4f}, p = {p_chi:.4f}, dof = {dof}")
print(f"→ {'Significant' if p_chi < 0.05 else 'Not significant'} at α = 0.05")

# ── 6b) Mann-Whitney U: Slope Active vs Placebo ──
s_active  = sub_known.loc[sub_known.study_arm == "Active",  slope_col].dropna()
s_placebo = sub_known.loc[sub_known.study_arm == "Placebo", slope_col].dropna()
u_stat, p_mw = mannwhitneyu(s_active, s_placebo, alternative="two-sided")
print(f"\nMann-Whitney U = {u_stat:.0f}, p = {p_mw:.4f}")
print(f"→ {'Significant' if p_mw < 0.05 else 'Not significant'} at α = 0.05")
print(f"  Active  median = {s_active.median():.3f} (n={len(s_active)})")
print(f"  Placebo median = {s_placebo.median():.3f} (n={len(s_placebo)})")

In [ ]:
# ── 6c) Chi-squared: Riluzole vs Rapid ──
ct_ril = pd.crosstab(df["riluzole_pre_t0"], df["rapid"])
chi2_ril, p_ril, dof_ril, _ = chi2_contingency(ct_ril)
print("Crosstab (Riluzole × Rapid):")
display(ct_ril)
print(f"\nχ² = {chi2_ril:.4f}, p = {p_ril:.4f}, dof = {dof_ril}")
print(f"→ {'Significant' if p_ril < 0.05 else 'Not significant'} at α = 0.05")

## 7) Save Summary Table

We consolidate the results into a CSV table for reference.

In [ ]:
# ── 7) Summary table ──
summary_rows = []

# Study arm
for arm in ["Active", "Placebo"]:
    sub = df[df.study_arm == arm]
    summary_rows.append({
        "Variable": "study_arm", "Group": arm,
        "N": len(sub), "Rapid_pct": round(100*sub.rapid.mean(), 1),
        "Slope_mean": round(sub[slope_col].mean(), 3),
        "Slope_median": round(sub[slope_col].median(), 3),
    })

# Riluzole
for ril in [0, 1]:
    sub = df[df.riluzole_pre_t0 == ril]
    summary_rows.append({
        "Variable": "riluzole", "Group": "Yes" if ril else "No",
        "N": len(sub), "Rapid_pct": round(100*sub.rapid.mean(), 1),
        "Slope_mean": round(sub[slope_col].mean(), 3),
        "Slope_median": round(sub[slope_col].median(), 3),
    })

# Interaction
for arm in ["Active", "Placebo"]:
    for ril in [0, 1]:
        sub = df[(df.study_arm == arm) & (df.riluzole_pre_t0 == ril)]
        if len(sub) > 5:
            summary_rows.append({
                "Variable": "arm×riluzole",
                "Group": f"{arm} + {'ril' if ril else 'no_ril'}",
                "N": len(sub), "Rapid_pct": round(100*sub.rapid.mean(), 1),
                "Slope_mean": round(sub[slope_col].mean(), 3),
                "Slope_median": round(sub[slope_col].median(), 3),
            })

summary = pd.DataFrame(summary_rows)
out_csv = os.path.join(OUT_TAB, "eda_treat_summary.csv")
summary.to_csv(out_csv, index=False)
display(summary)
print(f"\nSaved: {out_csv}")

## 8) Conclusions and Limitations

### Key Observations
- The **Active** group shows a slightly lower rapid progression rate than **Placebo**, but this difference needs to be evaluated statistically (Section 6)
- **Riluzole** alone does not show a clear association with slower progression — which is consistent with the literature (modest effect on survival, not necessarily on ALSFRS-R)
- The **Active + riluzole** combination has the lowest rapid progression rate, but the subgroup is relatively small

### Fundamental Limitations
1. **Active arm heterogeneity**: "Active" in PRO-ACT aggregates experimental drugs from multiple trials (lithium, dexpramipexole, ceftriaxone, etc.), most of which **did not demonstrate efficacy**. The observed signal is likely diluted.
2. **Confounders**: patients taking riluzole pre-baseline may have systematically different clinical profiles (indication bias)
3. **Correlation ≠ causation**: without causal inference techniques (propensity score matching, IPTW, CATE), it is not possible to conclude that treatment *causes* the difference in progression

### Possible Future Work
- **Propensity score matching** to isolate the riluzole effect, controlling for confounders
- **Heterogeneous Treatment Effects (CATE)** with causal forests to identify subgroups that benefit more
- **Per-trial analysis** (if original trial information is available) to separate different drugs